### BERT fine-tuning for document classification

In [87]:
import os
import re
import numpy as np 
from sklearn.metrics import accuracy_score

import transformers
from transformers import BertTokenizer, BertModel

import torch
from torch import cuda
from tqdm import tqdm
device = 'cuda' if cuda.is_available() else 'cpu'

device

'cpu'

- use X.txt and YL1.txt 

In [88]:
X = [line.strip() for line in open('X.txt').readlines()]
y = train_data = [int(line.strip()) for line in open('YL1.txt').readlines()]

len(X), len(y), max(y)

(46985, 46985, 6)

### An easy train/test split

In [135]:
train_X = X[:46000]
train_y = np.array(y[:46000])
test_X = X[46000:]
test_y = np.array(y[46000:])

len(train_X), len(train_y), len(test_X), len(test_y)

(46000, 46000, 985, 985)

In [136]:
# not needed for training or evaluation, but useful for mapping examples
labels = {
    0:'Computer Science',
    1:'Electrical Engineering',
    2:'Psychology',
    3:'Mechanical Engineering',
    4:'Civil Engineering',
    5:'Medical Science',
    6:'Biochemistry'
}

len(labels)

7

### Fine-tune BERT on the dataset

## Torch Datasets

In [137]:
class MultiLabelDataset(torch.utils.data.Dataset):

    def __init__(self, text, labels, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.text = text
        self.targets = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = self.text[index]
        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            pad_to_max_length=True,
            truncation=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.long)
        }

### Bert Class

In [138]:
class BERTClass(torch.nn.Module):
    def __init__(self, NUM_OUT):
        super(BERTClass, self).__init__()
                   
        self.l1 = BertModel.from_pretrained("bert-base-uncased")
#         self.pre_classifier = torch.nn.Linear(768, 256)
        self.classifier = torch.nn.Linear(768, NUM_OUT)
#         self.dropout = torch.nn.Dropout(0.5)
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
#         pooler = self.pre_classifier(pooler)
#         pooler = torch.nn.Tanh()(pooler)
#         pooler = self.dropout(pooler)
        output = self.classifier(pooler)
        output = self.softmax(output)
        return output

In [139]:
def loss_fn(outputs, targets):
    return torch.nn.CrossEntropyLoss()(outputs, targets)

def train(model, training_loader, optimizer):
    model.train()
    for data in tqdm(training_loader):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.long)

        outputs = model(ids, mask, token_type_ids)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return loss
    
def validation(model, testing_loader):
    model.eval()
    fin_targets=[]
    fin_outputs=[]
    with torch.no_grad():
        for data in tqdm(testing_loader):
            targets = data['targets']
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            outputs = model(ids, mask, token_type_ids)
            outputs = torch.nn.functional.softmax(outputs, dim = 1).cpu().detach()
            fin_outputs.extend(outputs)
            fin_targets.extend(targets)
    return torch.stack(fin_outputs), torch.stack(fin_targets)

### Tokenizer

In [140]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# what does the tokenizer do?
print(X[5])

tokenizer(
            train_X[5],
            add_special_tokens=True,
            max_length=128,
            padding=True,
            truncation=True,
            return_token_type_ids=True
        )

(Objective) In order to increase classification accuracy of tea-category identification (TCI) system, this paper proposed a novel approach. (Method) The proposed methods first extracted 64 color histogram to obtain color information, and 16 wavelet packet entropy to obtain the texture information. With the aim of reducing the 80 features, principal component analysis was harnessed. The reduced features were used as input to generalized eigenvalue proximal support vector machine (GEPSVM). Winner-takes-all (WTA) was used to handle the multiclass problem. Two kernels were tested, linear kernel and Radial basis function (RBF) kernel. Ten repetitions of 10-fold stratified cross validation technique were used to estimate the out-of-sample errors. We named our method as GEPSVM + RBF + WTA and GEPSVM + WTA. (Result) The results showed that PCA reduced the 80 features to merely five with explaining 99.90% of total variance. The recall rate of GEPSVM + RBF + WTA achieved the highest overall reca

{'input_ids': [101, 1006, 7863, 1007, 1999, 2344, 2000, 3623, 5579, 10640, 1997, 5572, 1011, 4696, 8720, 1006, 22975, 2072, 1007, 2291, 1010, 2023, 3259, 3818, 1037, 3117, 3921, 1012, 1006, 4118, 1007, 1996, 3818, 4725, 2034, 15901, 4185, 3609, 2010, 3406, 13113, 2000, 6855, 3609, 2592, 1010, 1998, 2385, 4400, 7485, 14771, 23077, 2000, 6855, 1996, 14902, 2592, 1012, 2007, 1996, 6614, 1997, 8161, 1996, 3770, 2838, 1010, 4054, 6922, 4106, 2001, 17445, 2098, 1012, 1996, 4359, 2838, 2020, 2109, 2004, 7953, 2000, 18960, 1041, 29206, 10175, 5657, 4013, 9048, 9067, 2490, 9207, 3698, 1006, 16216, 4523, 2615, 2213, 1007, 1012, 3453, 1011, 3138, 1011, 2035, 1006, 21925, 1007, 2001, 2109, 2000, 5047, 1996, 4800, 26266, 3291, 1012, 2048, 16293, 2015, 2020, 7718, 1010, 7399, 16293, 1998, 15255, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [143]:
MAX_LEN = 16
BATCH_SIZE = 32
EPOCHS = 3
NUM_OUT = 7 
LEARNING_RATE = 2e-05

training_data = MultiLabelDataset(train_X, torch.tensor(train_y, dtype = torch.long), tokenizer, MAX_LEN)
testing_data = MultiLabelDataset(test_X, torch.tensor(test_y, dtype = torch.long), tokenizer, MAX_LEN)

train_params = {'batch_size': BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }    

training_loader = torch.utils.data.DataLoader(training_data, **train_params)
testing_loader = torch.utils.data.DataLoader(testing_data, **test_params)

In [144]:
model = BERTClass(NUM_OUT)
model.to(device)    

optimizer = torch.optim.Adam(params =  model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    loss = train(model, training_loader, optimizer)
    print(f'Epoch: {epoch}, Loss:  {loss.item()}')  
    guess, targs = validation(model, testing_loader)
    guesses = torch.max(guess, dim=1).indices.cpu().numpy()
    targets = targs.cpu().numpy()
    print('arracy on test set {}'.format(accuracy_score(guesses, targets)))



Loading weights:   0%|                                                                         | 0/199 [00:00<?, ?it/s]

Loading weights:   1%|                 | 1/199 [00:00<00:00, 313.45it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|                  | 1/199 [00:00<00:02, 97.85it/s, Materializing param=embeddings.LayerNorm.bias]

Loading weights:   1%|▏               | 2/199 [00:00<00:02, 73.71it/s, Materializing param=embeddings.LayerNorm.weight]

Loading weights:   1%|▏               | 2/199 [00:00<00:03, 54.82it/s, Materializing param=embeddings.LayerNorm.weight]

Loading weights:   2%|      | 3/199 [00:00<00:03, 56.83it/s, Materializing param=embeddings.position_embeddings.weight]

Loading weights:   2%|      | 3/199 [00:00<00:04, 43.10it/s, Materializing param=embeddings.position_embeddings.weight]

Loading weights:   2%|    | 4/199 [00:00<00:03, 49.94it/s, Materializing param=embeddings.token_type_embeddings.weight]

Loading weights:   2%|    | 4/

Epoch: 0, Loss:  1.5074034929275513




  0%|                                                                                           | 0/31 [00:00<?, ?it/s]

  3%|██▋                                                                                | 1/31 [00:01<00:47,  1.60s/it]

  6%|█████▎                                                                             | 2/31 [00:03<00:47,  1.62s/it]

 10%|████████                                                                           | 3/31 [00:04<00:44,  1.59s/it]

 13%|██████████▋                                                                        | 4/31 [00:06<00:47,  1.75s/it]

 16%|█████████████▍                                                                     | 5/31 [00:08<00:43,  1.66s/it]

 19%|████████████████                                                                   | 6/31 [00:09<00:40,  1.62s/it]

 23%|██████████████████▋                                                                | 7/31 [00:11<00:38,  1.62s/it]

 26%|█████████████████████▍   

arracy on test set 0.650761421319797




  0%|                                                                                         | 0/1438 [00:00<?, ?it/s]C:\Users\cheik\AppData\Local\Temp\ipykernel_24564\3489272477.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'targets': torch.tensor(self.targets[index], dtype=torch.long)


  0%|                                                                               | 1/1438 [00:04<1:58:16,  4.94s/it]

  0%|                                                                               | 2/1438 [00:10<2:00:26,  5.03s/it]

  0%|▏                                                                              | 3/1438 [00:15<2:03:51,  5.18s/it]

  0%|▏                                                                              | 4/1438 [00:21<2:09:42,  5.43s/it]

  0%|▎                                                          

...

arracy on test set 0.6629441624365482




  0%|                                                                                         | 0/1438 [00:00<?, ?it/s]C:\Users\cheik\AppData\Local\Temp\ipykernel_24564\3489272477.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'targets': torch.tensor(self.targets[index], dtype=torch.long)


  0%|                                                                               | 1/1438 [00:05<2:03:07,  5.14s/it]

  0%|                                                                               | 2/1438 [00:10<1:59:32,  4.99s/it]

  0%|▏                                                                              | 3/1438 [00:14<1:58:53,  4.97s/it]

  0%|▏                                                                              | 4/1438 [00:19<1:57:24,  4.91s/it]

  0%|▎                                                          

Epoch: 2, Loss:  1.43990159034729




  0%|                                                                                           | 0/31 [00:00<?, ?it/s]

  3%|██▋                                                                                | 1/31 [00:01<00:37,  1.25s/it]

  6%|█████▎                                                                             | 2/31 [00:02<00:37,  1.28s/it]

 10%|████████                                                                           | 3/31 [00:03<00:33,  1.21s/it]

 13%|██████████▋                                                                        | 4/31 [00:04<00:32,  1.19s/it]

 16%|█████████████▍                                                                     | 5/31 [00:05<00:30,  1.17s/it]

 19%|████████████████                                                                   | 6/31 [00:07<00:29,  1.16s/it]

 23%|██████████████████▋                                                                | 7/31 [00:08<00:28,  1.17s/it]

 26%|█████████████████████▍   

arracy on test set 0.6548223350253807


### What does the BERT Tokenizer do?
The BERT Tokenizer tokenizes and case normalizes the text. Additionally it also does attention masking, which is the same as adding 0s to pad out the vectors.

### What loss function did you use? Why did you choose that loss function?
I used Categorical Cross Entropy due to the multi-class nature of the dataset. 

### (Edit) Try different batch sizes (e.g., 8 vs 16 vs 32). How does that affect your results?
Different batch sizes affect the amount of time it takes to train the data. 8 is the slowest speed, which ended up taking about 4 hours to train the data on my laptop, versus about 2 hours for a batch size of 32.

### Try another huggingface model (e.g., ELECTRA or RoBERTa) and compare results (you will need to change the name of the model and the tokenzier) How do the results compare to BERT?
I used the ELECTRA model to compare and I found that BERT was a little more accurate than ELECTRA, with both running on 3 epochs. From my understanding with higher epochs, ELECTRA would end up being more accurate than BERT.

### What is the power of fine-tuning (as opposed to pre-training)?
The power of fine-tuning is in the amount of time it takes for a given model to improve language classification. However, with pre-training it takes a much longer time to train the model and it would also require a much larger dataset compared to fine-tuning.